# 阶段 10 · OpenAI API 服务化 (Colab T4 GPU)

把"小鲸"包装成 OpenAI 兼容的 HTTP 服务, 用官方 openai 客户端调用它。

In [ ]:
# [Cell 1] 拉最新代码并装服务依赖
%cd /content/Lora
!git pull
!pip install -q fastapi uvicorn openai

In [ ]:
# [Cell 2] 后台启动服务 (日志写到 server.log, 不阻塞 notebook)
import subprocess, time
proc = subprocess.Popen(
    ["python", "src/deploy/openai_server.py", "--model", "outputs/lora_adapter", "--port", "8000"],
    stdout=open("server.log", "w"), stderr=subprocess.STDOUT,
)
print("服务启动中, 等待模型加载 (约 30~60 秒)...")

# 轮询日志, 直到出现 "接口就绪"
for _ in range(120):
    time.sleep(2)
    log = open("server.log").read()
    if "接口就绪" in log:
        print("✓ 服务已就绪")
        break
    if proc.poll() is not None:
        print("✗ 服务进程退出, 日志如下:\n", log)
        break
else:
    print("等待超时, 查看 server.log")

In [ ]:
# [Cell 3] 用官方 openai 客户端调用自己的服务
from openai import OpenAI

client = OpenAI(base_url="http://localhost:8000/v1", api_key="sk-no-key-needed")

resp = client.chat.completions.create(
    model="xiaojing",
    messages=[{"role": "user", "content": "你是谁？你是谁开发的？"}],
    temperature=0.3,
)
print("回答:", resp.choices[0].message.content)
print("用量:", resp.usage)

In [ ]:
# [Cell 4] 也可以用 curl 测试 (验证就是标准 HTTP 接口)
!curl -s http://localhost:8000/v1/models
print()
!curl -s http://localhost:8000/v1/chat/completions \
  -H "Content-Type: application/json" \
  -d '{"model":"xiaojing","messages":[{"role":"user","content":"用一句话介绍你自己"}]}'

## 生产路线对比: 用 vLLM 一行起服务 (可选)

合并模型后, vLLM 可直接提供同样的 OpenAI 接口 (高吞吐, 工业级):

```bash
!pip install vllm
!vllm serve models/xiaojing-merged --port 8000 --served-model-name xiaojing
```

注意: vLLM 在免费 T4 上对显存/算力要求较高, 可能需要加 `--max-model-len 2048 --dtype half` 等参数。学习阶段理解即可。

## 从外网访问 (可选)

Colab 服务默认只能内部访问。如需外网调用, 用内网穿透:

```python
!pip install pyngrok
from pyngrok import ngrok
ngrok.set_auth_token("你的ngrok_token")
url = ngrok.connect(8000)
print("公网地址:", url)   # 把它作为 base_url 即可远程调用
```